In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from Bio import SeqIO
from tqdm import tqdm
import json
import os
import logging
import numpy as np
import matplotlib.patches as patches
import random
from scipy.stats import mannwhitneyu
from sklearn.preprocessing import MinMaxScaler

# Import all our custom pipeline modules
from instanexus import preprocessing
from instanexus import assembly
from instanexus import visualization
from instanexus import helpers


# Set up logging to see the pipeline's progress
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
os.chdir('../../../../')

print(f"Current working directory: {os.getcwd()}")

In [ ]:
FIGURES_DIR = Path("figures")
print(FIGURES_DIR)

In [ ]:
# Path to the new raw data you want to test

INPUT_CSV = "inputs/bind3.csv"
METADATA_PATH = "json/sample_metadata.json"
CONTAMINANTS_PATH = "fasta/contaminants.fasta"
#COMPARISON_CSV = "inputs/ma.csv"
# polyclonal experiment
#RUN_NAME = Path(COMPARISON_CSV).stem
RUN_NAME = Path(INPUT_CSV).stem

# Metadata params
#MASS_ERR_LIMIT = 20
#MAX_IRT_ERROR = 60
#MIN_ENTROPY = 1
#PROSIT_FILTER = True
#Z_SCORE_THRESHOLD = -0.5

# Assembly params
ASSEMBLY_MODE = "dbg_weighted"
CHAIN = "heavy"
REFERENCE_MODE = True
KMER_SIZE = 6
MIN_OVERLAP = 2
SIZE_THRESHOLD = 0
CONFIDENCE_THRESHOLD = 0.8
MIN_LENGTH = 7
MAX_LENGTH = 20
FDR_THRESHOLD = 0.5
MIN_IDENTITY = 0.8
MAX_MISMATCHES = 0

# Clustering params
MIN_SEQ_ID = 0.85
COVERAGE = 0.8

In [ ]:
#base_output_folder = Path(BASE_OUTPUT_FOLDER) / RUN_NAME

# Build the unique experiment folder name
folder_name_parts = [f"{ASSEMBLY_MODE}"]

if CONFIDENCE_THRESHOLD is not None:
    folder_name_parts.append(f"c{CONFIDENCE_THRESHOLD}")

if "dbg" in ASSEMBLY_MODE:
    folder_name_parts.append(f"ks{KMER_SIZE}")

folder_name_parts.append(f"mo{MIN_OVERLAP}")
folder_name_parts.append(f"ts{SIZE_THRESHOLD}")

if REFERENCE_MODE:
    folder_name_parts.extend([f"mi{MIN_IDENTITY}", f"mm{MAX_MISMATCHES}"])

run_folder_name = "_".join(folder_name_parts)
#experiment_folder = base_output_folder / run_folder_name

run_id_str = f"[{RUN_NAME} @ {run_folder_name}]"

logger.info(f"Pipeline starting for run: {run_id_str}")

In [ ]:
sample_metadata = preprocessing.get_sample_metadata(
    run=RUN_NAME, 
    chain=CHAIN, 
    json_path=METADATA_PATH
)

In [ ]:
proteases = sample_metadata["proteases"]
protein = sample_metadata["protein"]
protein_norm = preprocessing.normalize_sequence(protein)

In [ ]:
print(f"Sample uses proteases: {proteases}")
print(f"Protein sequence length: {len(protein)} amino acids")
print(f"Normalized protein sequence: {protein_norm}")

In [ ]:
original_data = pd.read_csv(INPUT_CSV)

In [ ]:
original_data.columns

In [ ]:
cols_to_keep = [
    'experiment_name',
    'prediction_untokenised',
    'instanovo_token_log_probabilities',
    'calibrated_confidence',    
    'psm_q_value',
    'delta_mass_ppm',
    'Mass Error',               
    'is_missing_prosit_features', 
    'ion_match_intensity',
    'ion_matches',
    'iRT',
    'iRT error',
    'is_missing_irt_error',
    'predicted iRT',
    'margin',
    'entropy',
    'z-score'
    ]

data = original_data[cols_to_keep].copy()

In [ ]:
data.rename(columns={'calibrated_confidence': 'conf'}, inplace=True)

In [ ]:
data["protease"] = data["experiment_name"].apply(
    lambda name: preprocessing.extract_protease(name, proteases)
)

protease_col = data.pop("protease")
data.insert(data.columns.get_loc("prediction_untokenised") + 1, "protease", protease_col)

In [ ]:
data = data.dropna(subset=["prediction_untokenised"])

In [ ]:
data["cleaned_preds"] = data["prediction_untokenised"].apply(preprocessing.remove_modifications)

# move cleaned_preds next to prediction_untokenised
cleaned_preds_col = data.pop("cleaned_preds")
data.insert(data.columns.get_loc("prediction_untokenised") + 1, "cleaned_preds", cleaned_preds_col)

In [ ]:
cleaned_psms = data["cleaned_preds"].tolist()

In [ ]:
filtered_psms = preprocessing.filter_contaminants(
    cleaned_psms, RUN_NAME , CONTAMINANTS_PATH
)

In [ ]:
data = data[data["cleaned_preds"].isin(filtered_psms)]

In [ ]:
data.drop(columns=['prediction_untokenised'], inplace=True)

In [ ]:
data["mapped"] = data["cleaned_preds"].apply(
    lambda x: "True" if x in protein_norm else "False"
)

In [ ]:
data = data[data['cleaned_preds'].str.len() >= MIN_LENGTH]

data = data[data['cleaned_preds'].str.len() <= MAX_LENGTH]

In [ ]:
data['mapped'].value_counts()

### Adding quantification data

In [ ]:
def add_quantification_data(df_main, run_name, fdr_threshold, inputs_folder="inputs"):
    """
    Filters df_main by FDR, then looks for a quantification file ({run_name}_quant_scores.csv).
    Merges the abundance data into the filtered dataframe.
    """
    # 1. Filtro FDR (Inserito PRIMA del merge per pulizia)
    if fdr_threshold is not None:
        if "psm_q_value" in df_main.columns:
            initial_len = len(df_main)
            df_main = df_main[df_main['psm_q_value'] <= fdr_threshold].copy()
            logger.info(f"FDR Filter applied inside merge function: {initial_len} -> {len(df_main)} rows (<= {fdr_threshold})")
        else:
            logger.warning("FDR threshold provided but 'psm_q_value' column missing. Skipping filter.")

    # 2. Gestione Path
    quant_file_name = f"{run_name}_quant_scores.csv"
    quant_file_path = Path(inputs_folder) / quant_file_name
    
    if not quant_file_path.exists():
        logger.warning(f"Quantification file NOT FOUND: {quant_file_path}")
        logger.warning("Skipping abundance merging. 'peptide_abundance' will be missing.")
        return df_main

    logger.info(f"Found quantification file: {quant_file_path}")
    
    try:
        df_quant = pd.read_csv(quant_file_path)
        
        if "cleaned_preds" not in df_quant.columns or "total_abundance_norm" not in df_quant.columns:
            logger.warning(f"Quantification file format error. Missing columns in {quant_file_path}")
            return df_main

        df_quant_summed = df_quant.groupby('cleaned_preds', as_index=False)['total_abundance_norm'].sum()  
        df_quant_summed.rename(columns={'total_abundance_norm': 'peptide_abundance'}, inplace=True) 
        df_merged = pd.merge(df_main, df_quant_summed, on='cleaned_preds', how='left')
        df_merged['peptide_abundance'] = df_merged['peptide_abundance'].fillna(0)
        
        logger.info(f"Quantification data merged successfully. Output rows: {len(df_merged)}")
        return df_merged

    except Exception as e:
        logger.error(f"Error merging quantification data: {e}")
        return df_main

In [ ]:
FDR_THRESHOLD = 0.2
print(FDR_THRESHOLD)

In [ ]:
data_abundance = add_quantification_data(data, RUN_NAME, FDR_THRESHOLD)

In [ ]:
sequences = data_abundance['cleaned_preds'].tolist()

In [ ]:
print(KMER_SIZE, MIN_OVERLAP, SIZE_THRESHOLD)

In [ ]:
assembler = assembly.Assembler(
    mode="greedy",
    min_overlap=4,
    size_threshold=10,
    min_weight=2
    )

In [ ]:
scaffolds = assembler.run(sequences=sequences, df_full=data_abundance)

In [ ]:
mapped_scaffolds = visualization.process_protein_contigs_scaffold(
    scaffolds, protein_norm, 10, 0.7)

In [ ]:
print(RUN_NAME, CHAIN)

In [ ]:
mapped_scaffolds

In [ ]:
import importlib
importlib.reload(visualization)

In [ ]:
def mapping_sequences(
    mapped_sequences,
    prot_seq,
    category,
    run_name=None,      
    chain_type=None,    
    cdr_data=None,      
    config_json_path="json/colors.json",
    output_folder=".",
    output_file=None,
    show_figure=False,
):
    visualization.set_publication_style()
    
    if cdr_data and run_name and chain_type:
        print(f"DEBUG: Looking for CDRs -> Run: {run_name}, Chain: {chain_type}")
    else:
        print("DEBUG: No CDR info provided (Standard mode)")

    try:
        with open(config_json_path, 'r') as f:
            color_data = json.load(f)
        main_color = color_data.get(category, {}).get("scaffold", "#1f78b4")
    except Exception:
        main_color = "#1f78b4"

    fig_width, fig_height = visualization.get_figsize(width_ratio=3)
    common_height = 0.3
    track_spacing = 0.45
    base_y_offset = 0.6

    _, ax = plt.subplots(figsize=(fig_width, fig_height))

    ax.add_patch(patches.Rectangle(
        (0, 0), len(prot_seq), common_height,
        linewidth=0, facecolor='#e6f0ef', zorder=0
    ))

    cdr_colors = {"cdr1": "#FFB347", "cdr2": "#77DD77", "cdr3": "#89CFF0"}
    active_cdrs = {}

    if cdr_data and run_name and chain_type:
        if run_name in cdr_data:
            for entry in cdr_data[run_name]:
                if entry.get("chain", "").lower() == chain_type.lower():
                    active_cdrs = entry.get("cdrs", {})
                    print(f"DEBUG: Found CDRs: {list(active_cdrs.keys())}") # Debug
                    break
    
    tracks = {}
    colors = {
        "match": main_color,
        "mismatch": "#b30000",
        "D_to_N": "#000000",
        "E_to_Q": "#A8A29E",
    }

    for seq, mapping in tqdm(mapped_sequences, desc=f"Mapping {category}"):
        start_index, end_index, mismatches, _ = mapping
        placed = False
        for track_num in sorted(tracks.keys()):
            if not any(max(s, start_index) < min(e, end_index) for s, e in tracks[track_num]):
                tracks[track_num].append((start_index, end_index))
                current_track_num = track_num
                placed = True
                break
        if not placed:
            current_track_num = len(tracks)
            tracks[current_track_num] = [(start_index, end_index)]

        current_y = base_y_offset + (current_track_num * track_spacing)
        
        ax.add_patch(patches.Rectangle(
            (start_index, current_y), end_index - start_index, common_height,
            linewidth=0.8, edgecolor='white', facecolor=colors["match"], alpha=0.9, zorder=10
        ))

        for mismatch in mismatches:
            abs_index = start_index + mismatch
            if abs_index >= len(prot_seq) or mismatch >= len(seq): continue
            ref_aa = prot_seq[abs_index]
            query_aa = seq[mismatch]
            if query_aa == "D" and ref_aa == "N": mut_color = colors["D_to_N"]
            elif query_aa == "E" and ref_aa == "Q": mut_color = colors["E_to_Q"]
            else: mut_color = colors["mismatch"]
            
            ax.add_patch(patches.Rectangle(
                (abs_index, current_y), 1, common_height,
                linewidth=0, facecolor=mut_color, zorder=15
            ))

    max_track = len(tracks) if tracks else 0
    max_y = base_y_offset + (max_track * track_spacing) + 0.5
    
    for cdr_name, details in active_cdrs.items():
        if not details or "start" not in details: continue
        
        start = details["start"] - 1
        end = details["end"]
        width = end - start
        
        c_color = cdr_colors.get(cdr_name.lower(), "gray")
        
        cdr_rect = patches.Rectangle(
            (start, -0.5), width, max_y + 1, 
            facecolor=c_color, alpha=0.3, zorder=1, linewidth=0
        )
        ax.add_patch(cdr_rect)
        
        mid_point = (start + end) / 2
        ax.text(mid_point, max_y + 0.1, cdr_name.upper(), 
                ha='center', va='bottom', fontsize=9, fontweight='normal', 
                color='black', zorder=20)

    ax.set_xlim(0, len(prot_seq))
    y_upper_limit = max_y + 0.6 if active_cdrs else max_y
    ax.set_ylim(-0.1, y_upper_limit)
    
    ax.set_xlabel("Residue position")
    ax.set_yticks([])
    sns.despine(left=True)

    legend_patches = [
        patches.Patch(color=colors["match"], label=f"Match"),
        patches.Patch(color=colors["mismatch"], label="Mismatch"),
        patches.Patch(color=colors["D_to_N"], label="D \u2192 N"),
        patches.Patch(color=colors["E_to_Q"], label="E \u2192 Q"),
    ]
    ax.legend(handles=legend_patches, loc='upper center', bbox_to_anchor=(0.5, 1.25), ncol=4, frameon=False)

    plt.tight_layout()

    if output_file:
        try:
            os.makedirs(output_folder, exist_ok=True)
            save_path = os.path.join(output_folder, output_file)
            plt.savefig(save_path, format='svg', bbox_inches='tight')
            print(f"Saved: {save_path}")
        except Exception:
            plt.savefig(output_file, format='svg', bbox_inches='tight')

    if show_figure:
        plt.show()
    plt.close()

In [ ]:
with open("json/cdrs_antibodies.json", "r") as f:
    cdr_dict = json.load(f)

In [ ]:
mapping_sequences(
    mapped_scaffolds,
    protein_norm,
    category="binders",
    run_name=RUN_NAME,
    chain_type=CHAIN,
    cdr_data=cdr_dict,
    config_json_path="json/colors.json",
    output_folder=FIGURES_DIR,
    output_file=f"fig6c_{RUN_NAME}_{CHAIN}_{FDR_THRESHOLD}_scaffold_mapping.svg",
    show_figure=True
)

In [ ]:
df_mapped = visualization.create_dataframe_from_mapped_sequences(data=mapped_scaffolds)

In [ ]:
helpers.compute_assembly_statistics(
        df=df_mapped,
        sequence_type=f"scaffolds",
        output_folder="outputs/_polyclonal_analysis",
        reference=protein_norm,
        fdr_threshold=FDR_THRESHOLD 
    )

## Barplot coverage and composite score

In [ ]:
import importlib
importlib.reload(visualization)

In [ ]:
visualization.plot_barplot_composite_coverage_scores(
    csv_path="outputs/_summary_tables/dbg_weighted/best_results_Binders_dbg_weighted.csv",
    category="Binders",
    output_file_coverage="figures/fig6a_binders_coverage.svg",
    output_file_composite="figures/fig6b_binders_composite.svg",
    width_ratio=1
    )